# 13.1 Audio representation: waveform, spectrogram, MFCC

Audio becomes learnable when pressure over time is turned into local frequency evidence and compact spectral features.

Sampling turns air pressure into a vector, STFT windows expose local frequency evidence, and log filterbank projections compress that evidence into MFCC-like features.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1301)


def sine_wave(freq, duration, fs, amplitude=1.0, phase=0.0):
    n = np.arange(int(round(duration * fs)))
    x = amplitude * np.sin(2.0 * np.pi * freq * n / fs + phase)
    return x


def chirp_wave(start_freq, end_freq, duration, fs, amplitude=1.0):
    n = np.arange(int(round(duration * fs)))
    t = n / fs
    slope = (end_freq - start_freq) / duration
    phase = 2.0 * np.pi * (start_freq * t + 0.5 * slope * t * t)
    x = amplitude * np.sin(phase)
    return x


def rms(x):
    return float(np.sqrt(np.mean(np.square(x))))


def add_noise_for_snr(x, snr_db, seed):
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(size=len(x))
    target_noise_rms = rms(x) / (10.0 ** (snr_db / 20.0))
    noise = noise / (rms(noise) + 1e-12)
    y = x + target_noise_rms * noise
    return y


def frame_signal(x, n_fft, hop):
    if len(x) < n_fft:
        pad_width = n_fft - len(x)
        x = np.pad(x, (0, pad_width))
    starts = np.arange(0, len(x) - n_fft + 1, hop)
    frames = np.stack([x[start:start + n_fft] for start in starts])
    return frames


def stft_mag(x, n_fft, hop):
    frames = frame_signal(x, n_fft, hop)
    window = np.hanning(n_fft)
    spectrum = np.fft.rfft(frames * window[None, :], axis=1)
    magnitude = np.abs(spectrum)
    return magnitude


def hz_to_mel(freq):
    return 2595.0 * np.log10(1.0 + freq / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def mel_filterbank(fs, n_fft, n_mels):
    freqs = np.linspace(0.0, fs / 2.0, n_fft // 2 + 1)
    mel_edges = np.linspace(hz_to_mel(0.0), hz_to_mel(fs / 2.0), n_mels + 2)
    hz_edges = mel_to_hz(mel_edges)
    bank = np.zeros((n_mels, len(freqs)))
    for band in range(n_mels):
        left = hz_edges[band]
        center = hz_edges[band + 1]
        right = hz_edges[band + 2]
        rising = (freqs - left) / max(center - left, 1e-12)
        falling = (right - freqs) / max(right - center, 1e-12)
        bank[band] = np.maximum(0.0, np.minimum(rising, falling))
    return bank


def dct_matrix(n):
    rows = []
    for k in range(n):
        row = []
        for b in range(n):
            value = math.cos(math.pi * k * (2 * b + 1) / (2 * n))
            row.append(value)
        rows.append(row)
    return np.array(rows)


def build_audio_ladder(fs=8000):
    d1 = sine_wave(440.0, 0.45, fs)
    d2 = 0.65 * sine_wave(440.0, 0.50, fs) + 0.35 * sine_wave(660.0, 0.50, fs)
    d3_clean = 0.60 * sine_wave(440.0, 0.55, fs) + 0.35 * sine_wave(880.0, 0.55, fs)
    d3 = add_noise_for_snr(d3_clean, 12.0, 1303)
    d4 = np.concatenate([
        sine_wave(330.0, 0.20, fs),
        chirp_wave(360.0, 900.0, 0.25, fs),
        0.80 * sine_wave(550.0, 0.20, fs),
        0.65 * sine_wave(720.0, 0.15, fs),
    ])
    d5_clean = np.concatenate([
        0.70 * sine_wave(300.0, 0.25, fs),
        chirp_wave(320.0, 980.0, 0.35, fs),
        0.55 * sine_wave(440.0, 0.25, fs) + 0.25 * sine_wave(880.0, 0.25, fs),
        0.70 * sine_wave(660.0, 0.30, fs),
        chirp_wave(900.0, 420.0, 0.25, fs),
    ])
    d5 = add_noise_for_snr(d5_clean, 5.0, 1305)
    ladder = [
        {"name": "D1 single synthetic sine", "x": d1, "fs": fs, "targets": [440.0], "complexity": 1},
        {"name": "D2 two-tone command", "x": d2, "fs": fs, "targets": [440.0, 660.0], "complexity": 2},
        {"name": "D3 noisy two-tone", "x": d3, "fs": fs, "targets": [440.0, 880.0], "complexity": 3},
        {"name": "D4 chirp multi-segment", "x": d4, "fs": fs, "targets": [330.0, 550.0, 720.0], "complexity": 4},
        {"name": "D5 longer noisier phrase", "x": d5, "fs": fs, "targets": [300.0, 440.0, 660.0], "complexity": 5},
    ]
    return ladder


def dominant_frequencies(x, fs, n_fft=512, hop=128, count=3):
    mag = stft_mag(x, n_fft, hop)
    mean_mag = np.mean(mag, axis=0)
    mean_mag[0] = 0.0
    order = np.argsort(mean_mag)[::-1]
    freqs = order[:count] * fs / n_fft
    return freqs


def nearest_error(predicted, targets):
    predicted = np.array(predicted)
    errors = []
    for target in targets:
        error = float(np.min(np.abs(predicted - target)))
        errors.append(error)
    return float(np.mean(errors))


## The concept, built once (D1): waveform samples and STFT bins

A waveform sampled at rate $f_s$ is $x[n]$ with time $t=n/f_s$.  For the lesson's 440 Hz tone at $f_s=8000$, the first samples prove that a smooth pressure wave has become an array.

In [ ]:

fs = 8000
n = np.arange(int(0.01 * fs))
wave = np.sin(2.0 * np.pi * 440.0 * n / fs)
first_samples = np.round(wave[:5], 3)
energy = round(float(np.mean(wave * wave)), 3)

assert first_samples.tolist() == [0.0, 0.339, 0.637, 0.861, 0.982]
assert energy == 0.506

print("first samples", first_samples.tolist())
print("0.01 s mean squared energy", energy)


A short-time Fourier transform uses

$$X_m[k]=\sum_{n=0}^{N-1}x[n+mH]w[n]e^{-j2\pi kn/N}.$$

Bin $k$ corresponds to $k f_s/N$ Hz, so the method below returns waveform, STFT magnitude, log mel energy, and MFCC-like cosine coefficients.

In [ ]:

def represent_audio(x, fs, n_fft=256, hop=64, n_mels=4, floor=1e-6):
    magnitude = stft_mag(x, n_fft, hop)
    power = magnitude * magnitude
    bank = mel_filterbank(fs, n_fft, n_mels)
    energies = power @ bank.T
    log_energy = np.log(np.maximum(energies, floor))
    coeffs = log_energy @ dct_matrix(n_mels).T
    freqs = np.arange(n_fft // 2 + 1) * fs / n_fft
    return {
        "waveform": x,
        "magnitude": magnitude,
        "freqs": freqs,
        "log_energy": log_energy,
        "mfcc_like": coeffs,
    }

rep = represent_audio(sine_wave(440.0, 0.032, fs), fs, n_fft=256, hop=64, n_mels=4)
mean_mag = np.mean(rep["magnitude"], axis=0)
peak_bin = int(np.argmax(mean_mag))
peak_freq = float(rep["freqs"][peak_bin])
peak_mag = round(float(mean_mag[peak_bin]), 3)
spacing_256 = fs / 256
spacing_512 = fs / 512
floor_log = round(float(np.log(1e-6)), 1)

assert peak_bin == 14
assert peak_freq == 437.5
assert peak_mag == 63.489
assert spacing_256 == 31.25
assert spacing_512 == 15.625
assert floor_log == -13.8

print("nearest bin", peak_bin, peak_freq)
print("Hann FFT peak magnitude", peak_mag)
print("log floor", floor_log)


## The dataset ladder

Family F7 is audio-as-sequence, so the D1-D5 ladder is built inline from NumPy audio: single sine, two-tone, noisy two-tone, chirp/multi-segment, and longer/noisier audio.

In [ ]:

ladder = build_audio_ladder(fs=8000)

for rung in ladder:
    x = rung["x"]
    duration = len(x) / rung["fs"]
    sample = np.round(x[:8], 3).tolist()
    print(rung["name"])
    print("  samples", x.shape)
    print("  duration", round(duration, 3))
    print("  targets", rung["targets"])
    print("  preview", sample)


## Run the SAME method across D1-D5

The same `represent_audio` method is applied to every rung, and the metric is peak-frequency reconstruction error.

In [ ]:

rows = []

for rung in ladder:
    rep = represent_audio(rung["x"], rung["fs"], n_fft=512, hop=128, n_mels=20)
    predicted = dominant_frequencies(rung["x"], rung["fs"], n_fft=512, hop=128, count=5)
    error = nearest_error(predicted, rung["targets"])
    rows.append({"name": rung["name"], "metric": error, "predicted": predicted, "rep": rep})

print("rung | peak-frequency reconstruction error (Hz)")
for row in rows:
    print(row["name"], "|", round(row["metric"], 3), "| peaks", np.round(row["predicted"], 1).tolist())


## Results visualization

The closing figure has two parts: waveform/spectrogram panels for every rung, then the metric-vs-complexity curve.

In [ ]:

fig, axes = plt.subplots(5, 2, figsize=(10, 12))
metrics = []

for idx, rung in enumerate(ladder):
    row = rows[idx]
    x = rung["x"]
    fs = rung["fs"]
    rep = row["rep"]
    time = np.arange(len(x)) / fs
    axes[idx, 0].plot(time, x, linewidth=0.8)
    axes[idx, 0].set_title(rung["name"] + " waveform")
    axes[idx, 0].set_xlabel("seconds")
    image = axes[idx, 1].imshow(
        20.0 * np.log10(rep["magnitude"].T + 1e-6),
        origin="lower",
        aspect="auto",
        cmap="magma",
    )
    axes[idx, 1].set_title("spectrogram")
    axes[idx, 1].set_ylabel("frequency bin")
    metrics.append(row["metric"])

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), metrics, marker="o")
plt.xlabel("complexity rung")
plt.ylabel("frequency error (Hz)")
plt.title("Representation metric vs complexity")
plt.grid(True)
plt.show()


## Pitfall on D5: long STFT windows smear onsets

A long window improves frequency resolution but violates local stationarity on abrupt segments.  On the hardest rung, compare transient localization with a long window and a shorter window, while keeping the log-energy floor.

In [ ]:

d5 = ladder[-1]
long_rep = represent_audio(d5["x"], d5["fs"], n_fft=1024, hop=256, n_mels=20, floor=1e-12)
short_rep = represent_audio(d5["x"], d5["fs"], n_fft=256, hop=64, n_mels=20, floor=1e-6)
long_flux = np.mean(np.abs(np.diff(long_rep["magnitude"], axis=0)), axis=1)
short_flux = np.mean(np.abs(np.diff(short_rep["magnitude"], axis=0)), axis=1)
long_spread = int(np.sum(long_flux > 0.35 * np.max(long_flux)))
short_spread = int(np.sum(short_flux > 0.35 * np.max(short_flux)))
empty_log_without_floor = -np.inf
empty_log_with_floor = float(np.log(1e-6))

print("wrong long-window onset spread", long_spread)
print("fixed short-window onset spread", short_spread)
print("without floor", empty_log_without_floor)
print("with floor", round(empty_log_with_floor, 1))


## Evaluate it + Practice

- Metric: mean nearest peak-frequency reconstruction error in Hz across D1-D5
- No-skill baseline: guess the most common/simple output and compare against the ladder metric.
- Cheap sanity check: a 440 Hz D1 tone should peak at the 437.5 Hz bin for a 256-point FFT at 8000 Hz
- Ablation: remove the Hann/STFT representation and use raw samples only; the frequency error should rise or become undefined
- Failure signals: peaks at DC, infinite log energies, or onset flux smeared across many frames

Practice prompts:

**Practice.** Change D3's SNR and predict how the peak-frequency error should move.

**Practice.** Increase n_mels from 20 to 40 and inspect the MFCC-like coefficients.

**Practice.** Try a hop of 32 vs 128 and compare onset spread on D5.